# Surrogate Fidelity — public paper figures

This notebook regenerates the revised paper's public-data figures and tables from a clean checkout. The numerical specification is the canonical user-segment, pairwise-complete release analysis in `results/f_table.tsv`; the finite-extreme table is a sensitivity analysis and is never mixed into headline results.

The notebook intentionally excludes CKA and other analyses that require unreleased hidden states or analysis code. It writes paper-ready outputs below `paper_outputs/` by default. Set `SURROGATE_PAPER_OUTPUT_DIR` to a paper source directory to write there directly.


In [ ]:
import importlib.util
import os
from pathlib import Path
import shutil
import sys

import matplotlib
matplotlib.use("Agg")
import matplotlib.colors as mcolors
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch
import numpy as np
import pandas as pd
from scipy.special import logsumexp
from scipy.stats import gaussian_kde

try:
    from IPython.display import display
except ImportError:
    def display(value: object) -> None:
        print(value)


# Repository and output locations. Jupyter kernels need not start inside
# the checkout, so also resolve an editable installation of `surrogate`.
def _find_repo_root() -> Path:
    candidates: list[Path] = []
    override = os.environ.get("SURROGATE_REPO_ROOT")
    if override:
        candidates.append(Path(override).expanduser().resolve())

    cwd = Path.cwd().resolve()
    candidates.extend((cwd, *cwd.parents))
    package_spec = importlib.util.find_spec("surrogate")
    if package_spec is not None and package_spec.submodule_search_locations:
        candidates.extend(
            Path(location).resolve().parent
            for location in package_spec.submodule_search_locations
        )

    seen: set[Path] = set()
    for candidate in candidates:
        if candidate in seen:
            continue
        seen.add(candidate)
        if (candidate / "results" / "f_table.tsv").is_file():
            return candidate
    raise FileNotFoundError(
        "Could not locate the surrogate checkout. Install it with `pip install -e .`, "
        "start the kernel inside the checkout, or set SURROGATE_REPO_ROOT."
    )


REPO_ROOT = _find_repo_root()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
RESULTS_DIR = REPO_ROOT / "results"
OUTPUT_ROOT = Path(
    os.environ.get("SURROGATE_PAPER_OUTPUT_DIR", REPO_ROOT / "paper_outputs")
).resolve()
FIGURES_DIR = OUTPUT_ROOT / "figures" / "0625_cameraready"
TABLES_DIR = OUTPUT_ROOT / "tables"
FIGURES_DIR.mkdir(parents=True, exist_ok=True)
TABLES_DIR.mkdir(parents=True, exist_ok=True)
WRITE_PNG_PREVIEWS = os.environ.get("SURROGATE_PAPER_PNG_PREVIEWS", "0") == "1"


# One semantic palette for the paper.
PALETTE = {
    "F_pred": "#584486",       # violet: black-box prediction fidelity
    "F_attr": "#0E9AAB",       # teal: black-box attribution fidelity
    "F_mag": "#CE9955",        # amber: representation magnitude
    "F_align": "#984D6A",      # rose: representation alignment
    "F_attn": "#92A6C1",       # steel: attention family
    "F_cross": "#82CDD1",      # pale teal: cross-level fidelity
    "control": "#9A958F",      # warm gray: controls and nulls
    "class_false": "#2E2E38",  # ink
    "class_true": "#CE9955",   # amber
}
ATTENTION_LINESTYLES = {
    "F_attn_mean": "-",
    "F_attn_max": "--",
    "F_attn_rollout": ":",
}

OPEN_MODELS = [
    "qwen2.5-0.5b-instruct",
    "qwen2.5-3b-instruct",
    "llama-3.1-8b-instruct",
    "qwen2.5-7b-instruct",
    "qwen2.5-14b-instruct",
]
HOSTED_MODELS = [
    "llama3.1-70b-instruct",
    "llama3.3-70b-instruct",
    "llama4-maverick-17b-128e-instruct",
    "gpt-4o",
    "gpt-4-1",
    "gemini-2-5-flash-lite-vertex",
]
MODELS = OPEN_MODELS + HOSTED_MODELS
MODEL_LABELS = {
    "qwen2.5-0.5b-instruct": "Qwen-0.5B",
    "qwen2.5-3b-instruct": "Qwen-3B",
    "llama-3.1-8b-instruct": "Llama-8B",
    "qwen2.5-7b-instruct": "Qwen-7B",
    "qwen2.5-14b-instruct": "Qwen-14B",
    "llama3.1-70b-instruct": "Llama-70B",
    "llama3.3-70b-instruct": "Llama-3.3-70B",
    "llama4-maverick-17b-128e-instruct": "Maverick",
    "gpt-4o": "GPT-4o",
    "gpt-4-1": "GPT-4.1",
    "gemini-2-5-flash-lite-vertex": "Gemini Flash",
}

plt.rcParams.update({
    "font.family": "DejaVu Sans",
    "font.size": 8.5,
    "axes.titlesize": 10,
    "axes.labelsize": 9,
    "legend.fontsize": 7,
    "xtick.labelsize": 7.5,
    "ytick.labelsize": 7.5,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "pdf.fonttype": 42,
    "ps.fonttype": 42,
})


def _oklch_to_srgb(lightness: float, chroma: float, hue: float) -> tuple[float, float, float]:
    """Convert one OKLCH color to clipped display sRGB."""
    angle = np.deg2rad(hue)
    a = chroma * np.cos(angle)
    b = chroma * np.sin(angle)
    l_ = lightness + 0.3963377774 * a + 0.2158037573 * b
    m_ = lightness - 0.1055613458 * a - 0.0638541728 * b
    s_ = lightness - 0.0894841775 * a - 1.2914855480 * b
    l, m, s = l_**3, m_**3, s_**3
    linear = np.array([
        4.0767416621 * l - 3.3077115913 * m + 0.2309699292 * s,
        -1.2684380046 * l + 2.6097574011 * m - 0.3413193965 * s,
        -0.0041960863 * l - 0.7034186147 * m + 1.7076147010 * s,
    ])
    rgb = np.where(
        linear <= 0.0031308,
        12.92 * linear,
        1.055 * np.maximum(linear, 0.0) ** (1.0 / 2.4) - 0.055,
    )
    return tuple(np.clip(rgb, 0.0, 1.0))


def _gamut_mapped_oklch(lightness: float, chroma: float, hue: float) -> tuple[float, float, float]:
    """Reduce chroma at fixed lightness and hue until the color is in gamut."""
    low, high = 0.0, chroma
    for _ in range(24):
        candidate = (low + high) / 2.0
        angle = np.deg2rad(hue)
        a, b = candidate * np.cos(angle), candidate * np.sin(angle)
        l_ = lightness + 0.3963377774 * a + 0.2158037573 * b
        m_ = lightness - 0.1055613458 * a - 0.0638541728 * b
        s_ = lightness - 0.0894841775 * a - 1.2914855480 * b
        l, m, s = l_**3, m_**3, s_**3
        linear = np.array([
            4.0767416621 * l - 3.3077115913 * m + 0.2309699292 * s,
            -1.2684380046 * l + 2.6097574011 * m - 0.3413193965 * s,
            -0.0041960863 * l - 0.7034186147 * m + 1.7076147010 * s,
        ])
        if np.all((linear >= 0.0) & (linear <= 1.0)):
            low = candidate
        else:
            high = candidate
    return _oklch_to_srgb(lightness, low, hue)


def fidelity_ramp(name: str, hue: float, chroma_scale: float = 1.0) -> mcolors.ListedColormap:
    """Create a fixed-lightness-profile OKLCH ramp for an R² heatmap."""
    positions = np.linspace(0.0, 1.0, 256)
    lightness = 0.97 - 0.77 * positions
    chroma = 0.106 * chroma_scale * np.sin(np.pi * positions) ** 0.75
    colors = [
        _gamut_mapped_oklch(float(L), float(C), hue)
        for L, C in zip(lightness, chroma)
    ]
    return mcolors.ListedColormap(colors, name=name)


CMAP_PRED = fidelity_ramp("f_pred_oklch", 295.0)
CMAP_ATTR = fidelity_ramp("f_attr_oklch", 208.0)
CMAP_MAG = fidelity_ramp("f_mag_oklch", 70.0)
CMAP_ALIGN = fidelity_ramp("f_align_oklch", 350.0)
CMAP_GAP = mcolors.LinearSegmentedColormap.from_list(
    "fidelity_gap", [PALETTE["F_attr"], "#F7F6F4", PALETTE["F_pred"]]
)


def save_figure(figure: plt.Figure, filename: str) -> Path:
    """Save one stable, tightly cropped paper PDF."""
    path = FIGURES_DIR / filename
    figure.savefig(
        path,
        bbox_inches="tight",
        pad_inches=0.04,
        metadata={
            "Creator": "notebooks/paper_figures.ipynb",
            "CreationDate": None,
            "ModDate": None,
        },
    )
    if WRITE_PNG_PREVIEWS:
        figure.savefig(path.with_suffix(".png"), bbox_inches="tight", pad_inches=0.04, dpi=180)
    plt.close(figure)
    print(path.relative_to(OUTPUT_ROOT))
    return path


print(f"Repository: {REPO_ROOT}")
print(f"Outputs:    {OUTPUT_ROOT}")


## Load and validate the released tables

All headline binary results below come from the canonical pairwise-complete table. The assertions make accidental use of the finite-extreme sensitivity table, all-dialog scope, or ANLI entailment–neutral contrast fail loudly.


In [ ]:
F_TABLE = pd.read_csv(RESULTS_DIR / "f_table.tsv", sep="\t")
RACE_RV = pd.read_csv(RESULTS_DIR / "race_rv.tsv", sep="\t")
LAYER_CONTROL = pd.read_csv(
    REPO_ROOT / "layer_controls" / "layer_control_fidelity.tsv", sep="\t"
)

assert set(F_TABLE["scope"]) == {"user"}
assert set(F_TABLE["api_infinity_policy"]) == {"pairwise_complete"}
assert set(F_TABLE.loc[F_TABLE["benchmark"].str.startswith("anli"), "contrast"]) == {
    "entailment_contradiction"
}
assert set(RACE_RV["scope"]) == {"user"}
assert set(LAYER_CONTROL["scope"]) == {"user"}


def canonical_rows(
    *,
    benchmark: str,
    pregrouper: str,
    metric: str | None = None,
    statistic: str = "pearson_r2",
) -> pd.DataFrame:
    contrast = "entailment_contradiction" if benchmark.startswith("anli") else "canonical"
    mask = (
        F_TABLE["benchmark"].eq(benchmark)
        & F_TABLE["pregrouper"].eq(pregrouper)
        & F_TABLE["scope"].eq("user")
        & F_TABLE["contrast"].eq(contrast)
        & F_TABLE["statistic"].eq(statistic)
        & F_TABLE["availability_status"].eq("available")
        & F_TABLE["api_infinity_policy"].eq("pairwise_complete")
    )
    if metric is not None:
        mask &= F_TABLE["metric"].eq(metric)
    return F_TABLE.loc[mask].copy()


BOOLQ_R2 = canonical_rows(benchmark="boolq", pregrouper="sentence")
assert len(BOOLQ_R2.loc[BOOLQ_R2["metric"].eq("F_pred")]) == 55
assert len(BOOLQ_R2.loc[BOOLQ_R2["metric"].eq("F_attr")]) == 55
print("Canonical BoolQ pair rows:", len(BOOLQ_R2))


## Main summary table

In [ ]:
METRIC_LABELS = {
    "F_pred": r"$F_{\mathrm{pred}}$",
    "F_attr": r"$F_{\mathrm{attr}}$",
    "F_attn_mean": r"$F_{\mathrm{attn}}^{\mathrm{mean}}$",
    "F_attn_max": r"$F_{\mathrm{attn}}^{\mathrm{max}}$",
    "F_attn_rollout": r"$F_{\mathrm{attn}}^{\mathrm{rollout}}$",
    "F_mag": r"$F_{\mathrm{mag}}$",
    "F_align": r"$F_{\mathrm{align}}$",
    "F_mag_to_attr": r"$F_{\mathrm{mag}\to|\mathrm{attr}|}$",
    "F_align_to_attr": r"$F_{\mathrm{align}\to\mathrm{attr}}$",
    "F_attn_mean_to_attr": r"$F_{\mathrm{attn}\to\mathrm{attr}}^{\mathrm{mean}}$",
    "F_attn_max_to_attr": r"$F_{\mathrm{attn}\to\mathrm{attr}}^{\mathrm{max}}$",
    "F_attn_rollout_to_attr": r"$F_{\mathrm{attn}\to\mathrm{attr}}^{\mathrm{rollout}}$",
}

SUMMARY_GROUPS = [
    ("Black-box", ["F_pred", "F_attr"]),
    ("Representation-level", [
        "F_attn_mean", "F_attn_max", "F_attn_rollout", "F_mag", "F_align",
    ]),
    ("Mechanistic-to-causal", [
        "F_mag_to_attr", "F_align_to_attr", "F_attn_mean_to_attr",
        "F_attn_max_to_attr", "F_attn_rollout_to_attr",
    ]),
]


def _triplet(values: pd.Series) -> str:
    finite = pd.to_numeric(values, errors="coerce").dropna().to_numpy(dtype=float)
    if not len(finite):
        return "---"
    low, median, high = np.min(finite), np.median(finite), np.max(finite)
    return f"{low:.3f} / {median:.3f} / {high:.3f}"


summary_records = []
for group, metrics in SUMMARY_GROUPS:
    for metric in metrics:
        rows = BOOLQ_R2.loc[BOOLQ_R2["metric"].eq(metric)]
        symmetric_black_box = metric in {"F_pred", "F_attr"}
        representation_only = metric in {
            "F_attn_mean", "F_attn_max", "F_attn_rollout", "F_mag", "F_align"
        }
        broad_pops = (
            {"open_open", "open_hosted", "hosted_hosted"}
            if symmetric_black_box
            else {"open_open", "open_hosted"}
        )
        summary_records.append({
            "Group": group,
            "Metric": METRIC_LABELS[metric],
            "Open→Open": _triplet(rows.loc[rows["pair_population"].eq("open_open"), "f_point"]),
            "Open→Hosted": (
                "---" if representation_only else
                _triplet(rows.loc[rows["pair_population"].eq("open_hosted"), "f_point"])
            ),
            "All↔All / Open→All": (
                "---" if representation_only else
                _triplet(rows.loc[rows["pair_population"].isin(broad_pops), "f_point"])
            ),
        })

SUMMARY_TABLE = pd.DataFrame(summary_records)
display(SUMMARY_TABLE)

latex_lines = [
    r"% Canonical user-segment, pairwise-complete Pearson R^2; entries are min / median / max.",
    r"% The final column is All<->All for black-box rows and Open->All for cross-level rows.",
    r"\begin{tabular}{llccc}",
    r"\toprule",
    r"Group & Metric & Open$\to$Open & Open$\to$Hosted & All$\leftrightarrow$All / Open$\to$All \\",
    r"\midrule",
]
last_group = None
for record in summary_records:
    if last_group is not None and record["Group"] != last_group:
        latex_lines.append(r"\midrule")
    latex_lines.append(
        f"{record['Group'] if record['Group'] != last_group else ''} & {record['Metric']} & "
        f"{record['Open→Open']} & {record['Open→Hosted']} & "
        f"{record['All↔All / Open→All']} \\\\" 
    )
    last_group = record["Group"]
latex_lines.extend([r"\bottomrule", r"\end{tabular}"])
(TABLES_DIR / "summary_pearson.tex").write_text("\n".join(latex_lines) + "\n")
print(TABLES_DIR / "summary_pearson.tex")


## BoolQ fidelity dashboard

Both triangles use the same fixed $[0,1]$ scale and identical OKLCH lightness profiles. The upper triangle is $F_{\mathrm{pred}}$; the lower is signed $F_{\mathrm{attr}}$. The companion gap map makes the access–validity difference explicit.


In [ ]:
def pair_grid(metric: str) -> np.ndarray:
    frame = BOOLQ_R2.loc[BOOLQ_R2["metric"].eq(metric)]
    grid = np.full((len(MODELS), len(MODELS)), np.nan, dtype=float)
    model_index = {model: index for index, model in enumerate(MODELS)}
    for row in frame.itertuples(index=False):
        first, second = model_index[row.model_s], model_index[row.model_t]
        grid[first, second] = grid[second, first] = float(row.f_point)
    return grid


PRED_GRID = pair_grid("F_pred")
ATTR_GRID = pair_grid("F_attr")
indices = np.indices(PRED_GRID.shape)
upper_pred = np.where(indices[0] < indices[1], PRED_GRID, np.nan)
lower_attr = np.where(indices[0] > indices[1], ATTR_GRID, np.nan)

fig, ax = plt.subplots(figsize=(4.15, 3.95))
ax.imshow(upper_pred, cmap=CMAP_PRED, vmin=0.0, vmax=1.0)
ax.imshow(lower_attr, cmap=CMAP_ATTR, vmin=0.0, vmax=1.0)
for row in range(len(MODELS)):
    for column in range(len(MODELS)):
        if row == column:
            continue
        value = PRED_GRID[row, column] if row < column else ATTR_GRID[row, column]
        if np.isfinite(value):
            ax.text(
                column, row, f"{value:.2f}", ha="center", va="center",
                fontsize=5.8, color="white" if value > 0.68 else "#17171C",
            )
ax.set_xticks(range(len(MODELS)), [MODEL_LABELS[m] for m in MODELS], rotation=48, ha="right")
ax.set_yticks(range(len(MODELS)), [MODEL_LABELS[m] for m in MODELS])
ax.tick_params(length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_title(r"Upper: $F_{\mathrm{pred}}$   ·   Lower: signed $F_{\mathrm{attr}}$", pad=7)
ax.legend(
    handles=[
        Patch(facecolor=PALETTE["F_pred"], label=r"$F_{\mathrm{pred}}$ (fixed $R^2\in[0,1]$)"),
        Patch(facecolor=PALETTE["F_attr"], label=r"$F_{\mathrm{attr}}$ (fixed $R^2\in[0,1]$)"),
    ],
    frameon=False, loc="upper left", bbox_to_anchor=(1.01, 1.0), borderaxespad=0,
)
save_figure(fig, "fig9c_combined_heatmap_cameraready.pdf")

gap = PRED_GRID - ATTR_GRID
gap_limit = float(np.nanmax(np.abs(gap)))
fig, ax = plt.subplots(figsize=(4.15, 3.95))
image = ax.imshow(gap, cmap=CMAP_GAP, norm=mcolors.TwoSlopeNorm(vmin=-gap_limit, vcenter=0.0, vmax=gap_limit))
for row in range(len(MODELS)):
    for column in range(len(MODELS)):
        if row == column:
            continue
        value = gap[row, column]
        ax.text(column, row, f"{value:+.2f}", ha="center", va="center", fontsize=5.8,
                color="white" if abs(value) > 0.16 else "#17171C")
ax.set_xticks(range(len(MODELS)), [MODEL_LABELS[m] for m in MODELS], rotation=48, ha="right")
ax.set_yticks(range(len(MODELS)), [MODEL_LABELS[m] for m in MODELS])
ax.tick_params(length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
colorbar = fig.colorbar(image, ax=ax, fraction=0.038, pad=0.025)
colorbar.set_label(r"$F_{\mathrm{pred}}-F_{\mathrm{attr}}$")
ax.set_title("Prediction–attribution gap")
save_figure(fig, "fig9d_fidelity_gap_cameraready.pdf")


## BoolQ cross-family contour panels

The open model is the locally evaluated Llama-3.1-8B, not the retained API serving-path diagnostic. Token aliases are aggregated with logsumexp before forming true-minus-false log-odds. Attribution uses original minus ablated log-odds and is restricted to shared user-message coordinates.


In [ ]:
BOOLQ_SEGMENTS = pd.read_csv(RESULTS_DIR / "boolq" / "sentence" / "segments.tsv.gz", sep="\t")
USER_KEYS = BOOLQ_SEGMENTS.loc[
    BOOLQ_SEGMENTS["message_role"].eq("user"), ["prompt_idx", "seg_idx"]
]
assert len(USER_KEYS) == 17_706


def _label_logsumexp(values: pd.Series) -> float:
    available = values.dropna().to_numpy(dtype=float)
    return float(logsumexp(available)) if len(available) else float("nan")


def load_boolq_signals(model: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    tokens = pd.read_csv(
        RESULTS_DIR / "boolq" / "sentence" / f"{model}_tokens.tsv.gz",
        sep="\t",
        dtype={"kind": "string", "answer": "string", "label": "string", "token": "string"},
    )
    tokens["label"] = tokens["label"].str.lower()
    tokens["seg_key"] = tokens["seg_idx"].fillna(-1).astype(int)
    grouped = (
        tokens.groupby(
            ["prompt_idx", "seg_key", "kind", "answer", "label"],
            sort=False, dropna=False, observed=True,
        )["logprob"]
        .agg(_label_logsumexp)
        .unstack("label")
        .reset_index()
    )
    grouped["logodds"] = grouped["true"] - grouped["false"]
    original = grouped.loc[
        grouped["kind"].eq("orig") & grouped["seg_key"].eq(-1),
        ["prompt_idx", "answer", "logodds"],
    ].drop_duplicates("prompt_idx")
    ablated = grouped.loc[
        grouped["kind"].eq("ablated"), ["prompt_idx", "seg_key", "logodds"]
    ].rename(columns={"seg_key": "seg_idx", "logodds": "ablated_logodds"})
    attribution = (
        ablated.merge(
            original.rename(columns={"logodds": "original_logodds"}),
            on="prompt_idx", how="inner", validate="many_to_one",
        )
        .merge(USER_KEYS, on=["prompt_idx", "seg_idx"], how="inner", validate="one_to_one")
    )
    attribution["attribution"] = (
        attribution["original_logodds"] - attribution["ablated_logodds"]
    )
    return original, attribution


def paired_boolq_signals(source_model: str, target_model: str) -> tuple[pd.DataFrame, pd.DataFrame]:
    source_prediction, source_attribution = load_boolq_signals(source_model)
    target_prediction, target_attribution = load_boolq_signals(target_model)
    prediction = source_prediction.merge(
        target_prediction[["prompt_idx", "logodds"]],
        on="prompt_idx", suffixes=("_source", "_target"), validate="one_to_one",
    )
    attribution = source_attribution.merge(
        target_attribution[["prompt_idx", "seg_idx", "attribution"]],
        on=["prompt_idx", "seg_idx"], suffixes=("_source", "_target"), validate="one_to_one",
    )
    prediction = prediction.replace([np.inf, -np.inf], np.nan).dropna(
        subset=["logodds_source", "logodds_target"]
    )
    attribution = attribution.replace([np.inf, -np.inf], np.nan).dropna(
        subset=["attribution_source", "attribution_target"]
    )
    return prediction, attribution


def _r_squared(frame: pd.DataFrame, x: str, y: str) -> float:
    correlation = float(frame[x].corr(frame[y]))
    return correlation * correlation


def _contour_panel(
    frame: pd.DataFrame,
    x: str,
    y: str,
    x_label: str,
    y_label: str,
    filename: str,
) -> tuple[int, float]:
    x_limits = tuple(np.quantile(frame[x], [0.005, 0.995]))
    y_limits = tuple(np.quantile(frame[y], [0.005, 0.995]))
    fig, ax = plt.subplots(figsize=(2.55, 2.45))
    ax.set_xlim(*x_limits)
    ax.set_ylim(*y_limits)
    for truth, color, label in [
        (False, PALETTE["class_false"], "False"),
        (True, PALETTE["class_true"], "True"),
    ]:
        selected = frame.loc[frame["answer"].str.lower().eq(str(truth).lower())]
        if len(selected) > 6_000:
            selected = selected.sample(6_000, random_state=42)
        values = selected[[x, y]].to_numpy(dtype=float).T
        density = gaussian_kde(values)
        xx, yy = np.meshgrid(
            np.linspace(*x_limits, 72), np.linspace(*y_limits, 72)
        )
        zz = density(np.vstack([xx.ravel(), yy.ravel()])).reshape(xx.shape)
        levels = np.quantile(zz[zz > 0], [0.55, 0.72, 0.84, 0.92, 0.97])
        ax.contour(xx, yy, zz, levels=np.unique(levels), colors=color, linewidths=1.0, alpha=0.78)
        ax.plot([], [], color=color, label=label)
    r2 = _r_squared(frame, x, y)
    ax.text(0.04, 0.96, f"$R^2={r2:.3f}$\n$n={len(frame):,}$", transform=ax.transAxes,
            ha="left", va="top")
    ax.set_xlabel(x_label)
    ax.set_ylabel(y_label)
    ax.legend(frameon=False, loc="lower right")
    ax.grid(color="#E4E1DE", linewidth=0.5)
    save_figure(fig, filename)
    return len(frame), r2


PREDICTION_PAIR, ATTRIBUTION_PAIR = paired_boolq_signals(
    "llama-3.1-8b-instruct", "gpt-4o"
)
prediction_check = _contour_panel(
    PREDICTION_PAIR, "logodds_source", "logodds_target",
    "Llama-3.1-8B log-odds", "GPT-4o log-odds",
    "fig10a_logodds_contour_cameraready.pdf",
)
attribution_check = _contour_panel(
    ATTRIBUTION_PAIR, "attribution_source", "attribution_target",
    "Llama-3.1-8B attribution", "GPT-4o attribution",
    "fig10b_ablation_contour_cameraready.pdf",
)
assert prediction_check[0] == 3_096 and np.isclose(prediction_check[1], 0.656005, atol=5e-6)
assert attribution_check[0] == 16_246 and np.isclose(attribution_check[1], 0.428388, atol=5e-6)


## Attribution decomposition

These panels compare the public Llama-3.1-8B and Qwen-2.5-14B representation scalars on user-message segments. Alignment is the signed projection cosine $w\cdot\Delta z/(‖w‖‖\Delta z‖)$; it is not the cosine between original and perturbed representations. The magnitude-to-attribution row uses absolute attribution magnitude, as indicated by $F_{\mathrm{mag}\to|\mathrm{attr}|}$.


In [ ]:
def load_open_segments(model: str) -> pd.DataFrame:
    frame = pd.read_csv(
        RESULTS_DIR / "boolq" / "sentence" / f"{model}_segment.tsv.gz", sep="\t"
    )
    return frame.loc[frame["message_role"].eq("user")].copy()


llama_segments = load_open_segments("llama-3.1-8b-instruct")
qwen_segments = load_open_segments("qwen2.5-14b-instruct")
REPRESENTATION_PAIR = llama_segments.merge(
    qwen_segments,
    on=["prompt_idx", "seg_idx"],
    suffixes=("_llama", "_qwen"),
    validate="one_to_one",
)

for suffix, width in [("llama", 4096), ("qwen", 5120)]:
    REPRESENTATION_PAIR[f"alignment_{suffix}"] = (
        REPRESENTATION_PAIR[f"w_dot_delta_z_postnorm_{suffix}"]
        / (
            REPRESENTATION_PAIR[f"w_norm_{suffix}"]
            * REPRESENTATION_PAIR[f"delta_norm_postnorm_{suffix}"]
        )
    )
    REPRESENTATION_PAIR[f"norm_contribution_{suffix}"] = (
        np.sqrt(width)
        * REPRESENTATION_PAIR[f"w_dot_z_pert_prenorm_{suffix}"]
        * (
            REPRESENTATION_PAIR[f"z_pert_norm_prenorm_{suffix}"]
            - REPRESENTATION_PAIR[f"z_orig_norm_prenorm_{suffix}"]
        )
        / (
            REPRESENTATION_PAIR[f"z_orig_norm_prenorm_{suffix}"]
            * REPRESENTATION_PAIR[f"z_pert_norm_prenorm_{suffix}"]
        )
    )


def _joint_distribution(
    frame: pd.DataFrame,
    x: str,
    y: str,
    label: str,
    color: str,
    cmap: mcolors.Colormap,
    filename: str,
) -> float:
    finite = frame[[x, y]].replace([np.inf, -np.inf], np.nan).dropna()
    x_limits = tuple(np.quantile(finite[x], [0.005, 0.995]))
    y_limits = tuple(np.quantile(finite[y], [0.005, 0.995]))
    figure = plt.figure(figsize=(2.55, 2.55))
    grid = figure.add_gridspec(2, 2, width_ratios=(4, 1), height_ratios=(1, 4), hspace=0.05, wspace=0.05)
    top = figure.add_subplot(grid[0, 0])
    joint = figure.add_subplot(grid[1, 0], sharex=top)
    right = figure.add_subplot(grid[1, 1], sharey=joint)
    joint.hist2d(
        finite[x], finite[y], bins=58, range=[x_limits, y_limits],
        cmap=cmap, norm=mcolors.LogNorm(),
    )
    diagonal_low = max(x_limits[0], y_limits[0])
    diagonal_high = min(x_limits[1], y_limits[1])
    joint.plot([diagonal_low, diagonal_high], [diagonal_low, diagonal_high],
               color=PALETTE["control"], linestyle="--", linewidth=0.8)
    top.hist(finite[x], bins=58, range=x_limits, density=True, histtype="step", color=color)
    right.hist(finite[y], bins=58, range=y_limits, density=True, histtype="step",
               color=color, orientation="horizontal", linestyle="--")
    r2 = _r_squared(finite, x, y)
    joint.text(0.04, 0.96, f"$R^2={r2:.3f}$", transform=joint.transAxes, ha="left", va="top")
    joint.set_xlabel(f"Llama-3.1-8B {label}")
    joint.set_ylabel(f"Qwen-2.5-14B {label}")
    top.set_title(label)
    top.axis("off")
    right.axis("off")
    save_figure(figure, filename)
    return r2


decomposition_checks = {
    "magnitude": _joint_distribution(
        REPRESENTATION_PAIR, "delta_norm_postnorm_llama", "delta_norm_postnorm_qwen",
        "‖Δz‖", PALETTE["F_mag"], CMAP_MAG,
        "fig8a_delta_norm_cameraready.pdf",
    ),
    "alignment": _joint_distribution(
        REPRESENTATION_PAIR, "alignment_llama", "alignment_qwen",
        r"$\cos(\Delta z,w)$", PALETTE["F_align"], CMAP_ALIGN,
        "fig8b_cos_dz_v_cameraready.pdf",
    ),
    "normalization": _joint_distribution(
        REPRESENTATION_PAIR, "norm_contribution_llama", "norm_contribution_qwen",
        "RMSNorm contribution", PALETTE["F_mag"], CMAP_MAG,
        "fig8c_norm_contrib_cameraready.pdf",
    ),
}
expected_decomposition = {"magnitude": 0.727953, "alignment": 0.250954, "normalization": 0.001200}
for name, expected in expected_decomposition.items():
    assert np.isclose(decomposition_checks[name], expected, atol=5e-6), (name, decomposition_checks[name])
print(decomposition_checks)


## Revised per-layer figure

The primary curves are actual, unnormalized Pearson $R^2$ values over ten open-model pairs. Models are aligned on a common relative-depth grid by linearly interpolating signed scores over decoder-block outputs; the embedding slot is excluded. Target and gap ribbons are pointwise prompt-cluster bootstrap confidence intervals, with the gap computed inside each shared bootstrap resample. Control bands are empirical readout ranges, not confidence intervals.

The grouped 9-vs-8 direction is the readout-compatible control matched to headline grouped-logsumexp $F_{\mathrm{attr}}$. The observation-pair permutation breaks shared `(prompt_idx, seg_idx)` coordinates while preserving each model's attribution distribution. The isotropic curve is compared with the separately plotted single-token attribution diagnostic, because it does not implement the grouped headline readout.


In [ ]:
from benchmark_scripts.plot_layer_controls import plot_summary

layer_rows = LAYER_CONTROL.loc[LAYER_CONTROL["summary_kind"].eq("relative_depth")]
endpoint = layer_rows.sort_values("relative_depth_index").iloc[-1]
layer_endpoint = {
    "F_pred": endpoint["grouped_logsumexp_prediction_mean_pair_pearson_r2"],
    "F_attr": endpoint["grouped_logsumexp_attribution_mean_pair_pearson_r2"],
    "gap": endpoint["prediction_minus_attribution_gap_mean_pair_pearson_r2"],
    "gap_ci": (
        endpoint["prediction_minus_attribution_gap_bootstrap_lower"],
        endpoint["prediction_minus_attribution_gap_bootstrap_upper"],
    ),
}
print(layer_endpoint)
layer_figure = FIGURES_DIR / "fig16_fpred_fattr_gap_cameraready.pdf"
plot_summary(
    str(REPO_ROOT / "layer_controls" / "layer_control_fidelity.tsv"),
    str(layer_figure),
)
# Preserve the filename currently included by the paper source.
shutil.copyfile(
    layer_figure,
    FIGURES_DIR / "fig16_e5_random_dir_overlay_1col_cameraready.pdf",
)


## Cross-benchmark summary

Binary/completion tasks report median pairwise Pearson $R^2$. ANLI uses entailment minus contradiction. RACE reports the centered multivariate RV coefficient over all six A–D margins. The public release does not support representation-level or cross-level RACE cells under that multivariate estimand, so those cells remain blank.


In [ ]:
BENCHMARK_COLUMNS = [
    ("BoolQ", "boolq", "sentence"),
    ("ANLI R1", "anli_r1", "sentence"),
    ("ANLI R2", "anli_r2", "sentence"),
    ("ANLI R3", "anli_r3", "sentence"),
    ("WinoGrande", "winogrande", "sentence"),
    ("BoolQ word", "boolq", "word"),
    ("LAMBADA", "lambada", "word"),
]
TABLE_METRICS = [metric for _, metrics in SUMMARY_GROUPS for metric in metrics]


def _benchmark_median(metric: str, benchmark: str, pregrouper: str) -> tuple[float | None, int]:
    rows = canonical_rows(
        benchmark=benchmark, pregrouper=pregrouper, metric=metric
    )
    if metric in {"F_pred", "F_attr"}:
        populations = {"open_open", "open_hosted", "hosted_hosted"}
    elif metric.endswith("_to_attr"):
        populations = {"open_open", "open_hosted"}
    else:
        populations = {"open_open"}
    values = pd.to_numeric(
        rows.loc[rows["pair_population"].isin(populations), "f_point"], errors="coerce"
    ).dropna()
    return (float(values.median()), len(values)) if len(values) else (None, 0)


cross_records = []
count_records = []
for metric in TABLE_METRICS:
    record = {"Metric": METRIC_LABELS[metric]}
    counts = {"Metric": METRIC_LABELS[metric]}
    for label, benchmark, pregrouper in BENCHMARK_COLUMNS:
        value, count = _benchmark_median(metric, benchmark, pregrouper)
        record[label] = "---" if value is None else f"{value:.3f}"
        counts[label] = count
    race_metric = {"F_pred": "F_pred_rv", "F_attr": "F_attr_rv"}.get(metric)
    if race_metric is None:
        record["RACE RV"] = "---"
        counts["RACE RV"] = 0
    else:
        race_values = pd.to_numeric(
            RACE_RV.loc[
                RACE_RV["representation"].eq("all_pairs")
                & RACE_RV["metric"].eq(race_metric),
                "f_point",
            ], errors="coerce"
        ).dropna()
        record["RACE RV"] = f"{race_values.median():.3f}"
        counts["RACE RV"] = len(race_values)
    cross_records.append(record)
    count_records.append(counts)

CROSS_BENCHMARK_TABLE = pd.DataFrame(cross_records)
CROSS_BENCHMARK_COUNTS = pd.DataFrame(count_records)

# Pair-count assertions keep partial public coverage visible instead of silently
# changing the population summarized by a table cell.
for record in count_records:
    metric_label = record["Metric"]
    metric = next(key for key, label in METRIC_LABELS.items() if label == metric_label)
    for label, _, _ in BENCHMARK_COLUMNS:
        if metric in {"F_pred", "F_attr"}:
            expected = 28 if label == "LAMBADA" else 55
        elif metric.endswith("_to_attr"):
            expected = 35 if label == "LAMBADA" else 50
        else:
            expected = 10
        assert record[label] == expected, (metric, label, record[label], expected)
    assert record["RACE RV"] == (55 if metric in {"F_pred", "F_attr"} else 0)

race_all_pairs = RACE_RV.loc[RACE_RV["representation"].eq("all_pairs")]
race_coverage_min = float(race_all_pairs["observation_coverage"].min())
race_coverage_max = float(race_all_pairs["observation_coverage"].max())
assert np.isclose(race_coverage_min, 0.149056921264781)
assert np.isclose(race_coverage_max, 1.0)

display(CROSS_BENCHMARK_TABLE)
print("Pair counts (availability is pair-specific):")
display(CROSS_BENCHMARK_COUNTS)

columns = [label for label, _, _ in BENCHMARK_COLUMNS] + ["RACE RV"]
lines = [
    r"% User-segment canonical medians. Binary/completion columns: Pearson R^2; RACE: centered RV.",
    r"\resizebox{\textwidth}{!}{%",
    r"\begin{tabular}{l" + "c" * len(columns) + "}",
    r"\toprule",
    "Metric & " + " & ".join(columns) + r" \\",
    r"\midrule",
]
for record in cross_records:
    lines.append(record["Metric"] + " & " + " & ".join(record[column] for column in columns) + r" \\")
lines.extend([
    r"\bottomrule",
    r"\end{tabular}%",
    r"}",
    r"\par\smallskip",
    r"{\footnotesize LAMBADA coverage is 28/55 black-box pairs and 35/50 "
    r"mechanistic-to-causal pairs (representation-level: 10/10). RACE "
    rf"reports all 55 model pairs with pair-specific observation coverage "
    rf"from {100 * race_coverage_min:.1f}\% to {100 * race_coverage_max:.0f}\%.}}",
])
(TABLES_DIR / "cross_benchmark_pearson_cameraready.tex").write_text("\n".join(lines) + "\n")
print(TABLES_DIR / "cross_benchmark_pearson_cameraready.tex")


## Public RACE fidelity heatmap

In [ ]:
def race_grid(metric: str) -> np.ndarray:
    frame = RACE_RV.loc[
        RACE_RV["representation"].eq("all_pairs") & RACE_RV["metric"].eq(metric)
    ]
    grid = np.full((len(MODELS), len(MODELS)), np.nan)
    model_index = {model: index for index, model in enumerate(MODELS)}
    for row in frame.itertuples(index=False):
        first, second = model_index[row.model_s], model_index[row.model_t]
        grid[first, second] = grid[second, first] = float(row.f_point)
    return grid


race_pred = race_grid("F_pred_rv")
race_attr = race_grid("F_attr_rv")
indices = np.indices(race_pred.shape)
fig, ax = plt.subplots(figsize=(4.15, 3.95))
ax.imshow(np.where(indices[0] < indices[1], race_pred, np.nan), cmap=CMAP_PRED, vmin=0.0, vmax=1.0)
ax.imshow(np.where(indices[0] > indices[1], race_attr, np.nan), cmap=CMAP_ATTR, vmin=0.0, vmax=1.0)
for row in range(len(MODELS)):
    for column in range(len(MODELS)):
        if row == column:
            continue
        value = race_pred[row, column] if row < column else race_attr[row, column]
        if np.isfinite(value):
            ax.text(column, row, f"{value:.2f}", ha="center", va="center", fontsize=5.8,
                    color="white" if value > 0.68 else "#17171C")
ax.set_xticks(range(len(MODELS)), [MODEL_LABELS[m] for m in MODELS], rotation=48, ha="right")
ax.set_yticks(range(len(MODELS)), [MODEL_LABELS[m] for m in MODELS])
ax.tick_params(length=0)
for spine in ax.spines.values():
    spine.set_visible(False)
ax.set_title(r"RACE RV · upper: $F_{\mathrm{pred}}$ · lower: $F_{\mathrm{attr}}$")
save_figure(fig, "fig13_race_rv_cameraready.pdf")


## Output inventory

In [ ]:
EXPECTED_OUTPUTS = [
    TABLES_DIR / "summary_pearson.tex",
    TABLES_DIR / "cross_benchmark_pearson_cameraready.tex",
    FIGURES_DIR / "fig8a_delta_norm_cameraready.pdf",
    FIGURES_DIR / "fig8b_cos_dz_v_cameraready.pdf",
    FIGURES_DIR / "fig8c_norm_contrib_cameraready.pdf",
    FIGURES_DIR / "fig9c_combined_heatmap_cameraready.pdf",
    FIGURES_DIR / "fig9d_fidelity_gap_cameraready.pdf",
    FIGURES_DIR / "fig10a_logodds_contour_cameraready.pdf",
    FIGURES_DIR / "fig10b_ablation_contour_cameraready.pdf",
    FIGURES_DIR / "fig13_race_rv_cameraready.pdf",
    FIGURES_DIR / "fig16_fpred_fattr_gap_cameraready.pdf",
    FIGURES_DIR / "fig16_e5_random_dir_overlay_1col_cameraready.pdf",
]
missing = [path for path in EXPECTED_OUTPUTS if not path.is_file() or path.stat().st_size == 0]
assert not missing, missing
print("Generated", len(EXPECTED_OUTPUTS), "paper outputs")
for path in EXPECTED_OUTPUTS:
    print(f"{path.relative_to(OUTPUT_ROOT)}\t{path.stat().st_size:,} bytes")
